# 표준 라이브러리: 파일 검색과 재현 가능한 실험

공학프로그래밍입문 · 2026학년도 1학기 · 핵심 활동 약 40~50분

**목표는 문법 암기보다 원리를 이해하고, 코딩 AI에 정확한 질문·지시를 하며, 결과를 검토하는 능력입니다.**
코드는 예측한 뒤 실행하고, 값·조건을 직접 바꾸며 확인합니다. AI 도구를 사용할 수 없으면 제시된 가상 답변을 동료와 검토합니다.

준비: 이전 주차의 변수·자료형과 셀 실행 방법

## 오늘 설명할 수 있어야 하는 것

- 직접 구현할 작업과 표준 기능을 사용할 작업을 구분한다.
- 검색 조건과 정렬 기준을 명시하고 작은 파일 목록으로 결과를 확인한다.


## 1. 먼저 원리를 읽기

라이브러리는 검증된 기능을 재사용하게 해 주지만, 기본 동작이 내 요구와 같은지는 확인해야 합니다.
`pathlib`은 경로, `json`은 데이터 교환, `random`은 난수 생성에 사용합니다. 파일 시스템의 검색 결과 순서는 보고서 순서라는 계약이 아닙니다. 확장자와 폴더 포함 여부도 명시하세요.


## 2. 실행 전 예측하기

이름이 `.csv`로 끝나는 폴더도 실습 데이터 파일로 집계해야 할까요? `rain.CSV`는 포함할까요?


아래 기록을 먼저 채운 뒤 다음 코드 셀을 실행하세요. 정답과 다르면 예측에 사용한 가정을 고칩니다.

- 내 예상 결과:
- 그렇게 생각한 이유:
- 실행 뒤 발견한 차이:


In [ ]:
from pathlib import Path
import tempfile
import json
import random

workspace = Path(tempfile.mkdtemp(prefix="ce100-files-"))
for name in ["b.csv", "a.CSV", "notes.txt"]:
    (workspace / name).write_text("synthetic practice\n", encoding="utf-8")
(workspace / "fake.csv").mkdir()
entries = sorted(p.name for p in workspace.iterdir())
print("파일과 폴더:", entries)
assert "fake.csv" in entries


### 관찰을 원리와 연결하기

확장자만 맞아도 폴더일 수 있습니다. 무엇을 집계할지 먼저 정하고 `is_file()` 같은 성질을 확인합니다. 여기서는 하위 폴더를 탐색하지 않습니다.


다음 셀의 입력이나 조건을 하나 바꾸세요. 바꾼 값, 예상한 변화, 실제 결과를 적습니다. 단순 재실행만으로 이 활동을 마치지 않습니다.


In [ ]:
names = sorted(p.name for p in workspace.iterdir()
               if p.is_file() and p.suffix.lower() == ".csv")
manifest_path = workspace / "manifest.json"
manifest_path.write_text(json.dumps(names, ensure_ascii=False), encoding="utf-8")
assert json.loads(manifest_path.read_text(encoding="utf-8")) == ["a.CSV", "b.csv"]
rng1, rng2 = random.Random(42), random.Random(42)
sample1 = [rng1.randint(0, 10) for _ in range(5)]
sample2 = [rng2.randint(0, 10) for _ in range(5)]
assert sample1 == sample2
print("같은 seed의 연습 관측:", sample1)


## 3. 작은 기능을 직접 완성하기

`list_csv(folder)`는 `Path`를 받아 바로 아래의 CSV **파일 이름**을 오름차순 리스트로 반환합니다. 확장자 대소문자는 구분하지 않고 하위 폴더는 탐색하지 않습니다.
빈 폴더는 빈 리스트를 반환합니다. 경로는 존재하는 폴더로 한정합니다.


먼저 입력·출력 예를 하나 더 직접 만드세요. 모르는 문법은 도움말이나 AI에 질문해도 됩니다. AI 코드 전체를 받기 전에 필요한 처리 단계를 자신의 말로 2~3줄 적으세요.

- 내가 정한 추가 입력과 예상 출력:
- 처리 단계:


In [ ]:
def list_csv(folder):
    # TODO: 파일 여부, 확장자, 탐색 범위, 정렬을 각각 확인하세요.
    raise NotImplementedError("검색 조건을 코드로 옮기세요")


### 공개 점검

아래 점검 셀을 실행합니다. **“미완료”는 통과가 아닙니다.** 위 함수를 구현한 뒤 다시 실행하세요.
실패하면 기대값을 바꾸기보다 입력·계약·구현 중 무엇이 다른지 조사합니다. 공개 시험만 통과했다고 모든 입력에 옳다는 뜻은 아닙니다.


In [ ]:
def check_answer(candidate):
    with tempfile.TemporaryDirectory() as directory:
        root = Path(directory)
        assert candidate(root) == []
        for name in ["z.csv", "a.CSV", "b.txt"]:
            (root / name).write_text("test", encoding="utf-8")
        (root / "folder.csv").mkdir()
        nested = root / "nested"
        nested.mkdir()
        (nested / "hidden.csv").write_text("test", encoding="utf-8")
        assert candidate(root) == ["a.CSV", "z.csv"], "검색 범위와 파일 여부를 확인하세요"
    print("공개 점검 통과: 빈 폴더·대소문자·정렬·하위 폴더 제외")

try:
    check_answer(list_csv)
except NotImplementedError:
    print("미완료: 위 TODO를 구현한 뒤 이 셀을 다시 실행하세요. 아직 점검을 통과하지 않았습니다.")


## 4. AI 제안을 검토하고 수정 지시하기

다음은 오류를 검토하기 위해 만든 **가상 AI 답변**입니다. 실제 AI 응답이나 정답이 아닙니다.
실행 전에 어떤 요구사항을 놓쳤는지 찾고, 반례가 그 문제를 어떻게 드러내는지 설명하세요.


In [ ]:
def ai_suggestion(folder):
    return sorted(p.name for p in folder.iterdir() if p.suffix.lower() == ".csv")

print("가상 제안:", ai_suggestion(workspace))
assert "fake.csv" in ai_suggestion(workspace)


### 검토 질문

파일명 규칙만 검사해서 폴더도 파일로 보고합니다. AI에게 실제로 시험할 수 있는 작은 디렉터리 구조를 제시하고, 포함·제외 목록을 함께 전달하세요.


- 위 제안이 어긴 요구사항:
- 오류를 드러내는 입력과 기대 결과:
- 한 문장 수정 지시:


### 코딩 AI에게 보낼 요청 작성

Path를 입력으로 받아 CSV 파일명만 반환하는 함수를 요청한다. 확장자 대소문자, 하위 폴더 탐색 여부, 폴더 제외, 출력 정렬, 빈 결과 정책을 명시하고 임시 폴더 시험을 요구한다.


위 문장을 그대로 복사하는 대신 오늘의 실제 입력·기대값·실패 조건을 넣어 요청문을 완성하세요.
아래 틀로 **구현 전 계획**을 먼저 요청하고, 받은 코드에는 새 반례를 하나 추가합니다.

```text
목적과 환경: Python / Google Colab에서 ... 하려고 한다.
입력: 자료형·구조·단위는 ... 이며, 예시는 ... 이다.
출력: ... 형식으로 ... 를 반환해야 한다.
조건: ... 는 포함하고 ... 는 제외한다. 잘못된 입력은 ... 로 처리한다.
검증: 정상 예 ... / 경계 예 ... / 반례 ... 의 기대 결과는 ... 이다.
먼저 모호한 조건을 짚고 처리 계획을 설명해 줘. 그 뒤 짧은 코드와 검증을 제시해 줘.
```

- 내가 보낸 최초 요청:
- 검토 후 보낸 수정 요청:
- AI 제안에서 채택·수정·거절한 부분과 근거:
- 내가 추가한 시험의 입력·기대값·실제값:


## 5. 새 상황에 옮겨 보기

하위 폴더까지 검색하도록 바뀌면 이름만 반환해도 충분할까요? 이름이 같은 두 파일을 구분할 출력 구조를 제안하세요.


이 활동은 한 가지 구현을 암기했는지가 아니라 요구사항이 달라졌을 때 무엇을 다시 질문하는지 확인합니다.


## 마무리 기록

- 오늘 사용한 핵심 원리를 자신의 말로 한 문장:
- 결과를 신뢰할 근거(시험 입력·기대값·실제값):
- 아직 확인하지 않은 조건 또는 적용 한계:
- AI 없이 설명할 수 있는 코드 한 줄과 그 역할:

**완료 기준:** 예측을 남겼고, 입력·조건을 바꿨고, TODO를 구현해 공개 점검을 통과했고, 새 반례와 수정 지시를 작성했으며, 결과의 한계를 설명할 수 있습니다.

[과목의 실습 방식과 점검 기준](../LEARNING_GUIDE.md)을 참고하세요.


## 선택 확장

sys.argv와 sys.path의 역할을 비교하고, 원본 예제의 shutil·glob·calendar·time은 필요한 작업이 생겼을 때 문서에서 찾아 사용합니다.


## 출처와 개편 범위

    기존 강의 노트북의 주제를 바탕으로 핵심 예제, 합성 연습 데이터, AI 검토 활동을 새로 구성했습니다.
    원본 파일명은 유지했고, 원본 Drive 링크와 개편 전·후 해시는 강의 폴더의 `materials-manifest.json`에 기록합니다.
    노트북 실행에 외부 AI API나 원본 로컬 ZIP은 필요하지 않습니다. 학습용 오류 제안은 반례를 확인하는 코드이며, 미완성 TODO와 구분합니다.
